# Exploratory Sanity Checks

Quick checks to verify the project scaffold is working:
- Data schemas
- Reward computation
- Candidate generation
- Metrics

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

## 1. Data Schemas

In [ ]:
from gui_grounding.data.schemas import ActionType, BBox, CandidateAction, GroundingSample

sample = GroundingSample(
    sample_id="demo_001",
    dataset_name="mind2web",
    split="train",
    image_path="data/raw/mind2web/screenshots/demo.png",
    instruction="Click the search button",
    action_type=ActionType.CLICK,
    target_bbox=BBox(x1=100, y1=200, x2=250, y2=240),
    click_point=(175, 220),
    website="example.com",
)

print(f"Sample: {sample.sample_id}")
print(f"Instruction: {sample.instruction}")
print(f"Target bbox: {sample.target_bbox.as_tuple()}")
print(f"Target center: {sample.target_bbox.center}")
print(f"Target area: {sample.target_bbox.area}")

## 2. Reward Computation

In [ ]:
from gui_grounding.reward.verifiable_reward import VerifiableRewardCalculator

calc = VerifiableRewardCalculator()

# Perfect prediction
result = calc.compute(
    sample_id="demo_001",
    pred_bbox=(100, 200, 250, 240),
    gt_bbox=(100, 200, 250, 240),
    pred_click=(175, 220),
    pred_action="click",
    gt_action="click",
)
print(f"Perfect prediction reward: {result.total_reward:.4f}")
print(f"Components: {result.components}")

# Partial prediction
result2 = calc.compute(
    sample_id="demo_002",
    pred_bbox=(120, 210, 260, 250),
    gt_bbox=(100, 200, 250, 240),
    pred_click=(190, 230),
    pred_action="click",
    gt_action="click",
)
print(f"\nPartial prediction reward: {result2.total_reward:.4f}")
print(f"IoU: {result2.components.iou:.4f}")

## 3. Candidate Generation + Reward Scoring

In [ ]:
from gui_grounding.reward.candidate_generator import CandidateGenerator

gen = CandidateGenerator(mode="dummy", num_candidates=8, seed=42)
candidates = gen.generate(sample)

print(f"Generated {len(candidates)} candidates:")
for c in candidates:
    reward = calc.compute(
        sample_id=c.candidate_id,
        pred_bbox=c.bbox.as_tuple() if c.bbox else None,
        gt_bbox=sample.target_bbox.as_tuple(),
        pred_click=c.click_point,
        pred_action=c.action_type,
        gt_action=sample.action_type,
    )
    print(f"  {c.candidate_id}: reward={reward.total_reward:.4f}, iou={reward.components.iou:.3f}")

## 4. Evaluation Metrics

In [ ]:
from gui_grounding.evaluation.metrics import compute_all_metrics

metrics = compute_all_metrics(
    pred_element_ids=["btn_1", "btn_2", "btn_3", "btn_4"],
    gt_element_ids=["btn_1", "btn_2", "btn_5", "btn_4"],
    pred_bboxes=[(10, 20, 100, 80), (50, 50, 150, 150), (0, 0, 50, 50), (200, 200, 300, 300)],
    gt_bboxes=[(10, 20, 100, 80), (60, 60, 140, 140), (200, 200, 300, 300), (200, 200, 300, 300)],
    pred_points=[(55, 50), (100, 100), (25, 25), (250, 250)],
    pred_actions=["click", "type", "click", "hover"],
    gt_actions=["click", "type", "select", "hover"],
)

print("Evaluation Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

## 5. Config Loading

In [ ]:
from gui_grounding.utils.config import load_config

cfg = load_config("train/sft_baseline.yaml")
print(f"Experiment: {cfg.experiment.name}")
print(f"Stage: {cfg.experiment.stage}")
print(f"LR: {cfg.training.learning_rate}")
print(f"Epochs: {cfg.training.num_epochs}")